# Six-source raw-hidden-state MLP baseline

Reuse **all seven domains' saved Qwen generations and answer-token hidden states** plus saved BLEURT-20 scores from `nvthaai/sala-g14-6-source-triviaqa-version-6-artifacts`. Train the same Mish/1024/dropout-0.2 MLP with BCE on the six original source datasets, but flatten and concatenate all 29 raw Qwen states instead of any SALA projection. TriviaQA remains held out. Set GPU T4 x2 and Internet on, and attach that private Kaggle dataset before Save & Run All.

In [ ]:
!git clone https://github.com/brainardphilemon/SALA.git /kaggle/working/SALA
%cd /kaggle/working/SALA
!git checkout b7f8f4b
!pip install -q -r requirements-kaggle.txt

In [ ]:
from pathlib import Path
SOURCE_NAME = 'sala_qwen25_g14_6src_triviaqa_v1'
domains = ['tqa', 'nq_open', 'sciq', 'popqa', 'web_questions', 'hotpotqa', 'triviaqa']
artifact_dirs = [p for p in Path('/kaggle/input').rglob(SOURCE_NAME) if p.name == SOURCE_NAME and (p / 'generation').is_dir()]
if len(artifact_dirs) != 1:
    raise RuntimeError(f'Expected one complete saved artifact set; found {artifact_dirs}')
ARTIFACT_DIR = artifact_dirs[0]
required = [*(ARTIFACT_DIR / 'generation' / d / f for d in domains for f in ('rows.jsonl', 'features.npy', 'manifest.json')), *(ARTIFACT_DIR / 'bleurt' / d / f for d in domains for f in ('scores.jsonl', 'manifest.json'))]
missing = [str(p) for p in required if not p.is_file()]
if missing:
    raise RuntimeError('Missing saved generation or BLEURT files:\n' + '\n'.join(missing))
print('Reusing all saved source and target generations:', ARTIFACT_DIR)
print('Validated', len(required), 'files; no Qwen or BLEURT rerun.')

In [ ]:
import subprocess, sys
OUTPUT_DIR = Path('/kaggle/working/sala_g14_6src_raw_concat')
command = [sys.executable, '/kaggle/working/SALA/sala_raw_concat.py', '--input-artifact-dir', str(ARTIFACT_DIR), '--output-dir', str(OUTPUT_DIR), '--bleurt-threshold', '0.5', '--epochs-erm', '40', '--seed', '42', '--train-device', 'cuda:0']
print('Running:', ' '.join(command))
subprocess.run(command, check=True)

In [ ]:
import json, pandas as pd
results = pd.read_csv(OUTPUT_DIR / 'results.csv')
summary = json.loads((OUTPUT_DIR / 'summary.json').read_text())
print('Rows:', len(results), 'MLP input:', summary['mlp_input_dimension'], 'threshold:', summary['sala_threshold'])
print(json.dumps(summary['target_metrics'], indent=2))
display(results.head())